# TP Jour 3 – Exploration des sources Bronze avec PySpark (UC-06, étapes 2 et 3)

**Retail Intelligence Platform** – IPSSI, Mastère Dev, Data & IA – 18/11/2026

Objectif : observer les données **avant** de les transformer. Pour chaque source retenue, on charge le fichier,
on affiche son schéma, un échantillon et le nombre de lignes, puis on cherche les valeurs manquantes, les
doublons et les valeurs atypiques. Aucune donnée n'est modifiée ici : on ne fait que **constater**.

Sources explorées en détail : `store_sales` (ventes magasin, la table de faits principale) et `item` (produits).
Les 8 tables retenues sont profilées automatiquement par `transformations/02_inventaire.py` (résumé en fin de notebook).

Environnement : Spark 4.3.0 en mode local (`local[*]`), Ubuntu 24.04 (WSL 2), données lues dans **HDFS**
(zone Bronze alimentée par `01_bronze.py` depuis le dépôt Raw du Jour 2).

In [1]:
import sys
sys.path.insert(0, "../transformations")      # paramètres et fonctions communes du pipeline
import config as C
import commun as U
from pyspark.sql import functions as F

spark, log = U.session("notebook_exploration")
print("Version Spark :", spark.version, "| master :", spark.sparkContext.master)
print("Zone Bronze  :", C.BRONZE)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 12:50:16 WARN Utils: Your hostname, retail-lake-vm, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/18 12:50:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 12:50:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-11-18 12:50:22 INFO Session Spark 4.3.0 (master=local[*]), lac=hdfs://localhost:9000/retail-lake


Version Spark : 4.3.0 | master : local[*]
Zone Bronze  : hdfs://localhost:9000/retail-lake/bronze/tpcxbb


## 1. Première lecture « naïve » : laisser Spark deviner

Le TP propose `inferSchema=True`. Essayons sur le fichier des ventes magasin du lot initial, **sans en-tête**
(les fichiers TPCx-BB n'ont pas de ligne de titres) et avec le séparateur `|`.

In [2]:
chemin = f"{U.chemin_bronze('initial', 'store_sales')}/store_sales.dat"
naif = spark.read.option("header", False).option("inferSchema", True).option("sep", "|").csv(chemin)
naif.printSchema()
naif.show(3)

root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: integer (nullable = true)
 |-- _c4: integer (nullable = true)
 |-- _c5: integer (nullable = true)
 |-- _c6: integer (nullable = true)
 |-- _c7: integer (nullable = true)
 |-- _c8: integer (nullable = true)
 |-- _c9: integer (nullable = true)
 |-- _c10: integer (nullable = true)
 |-- _c11: double (nullable = true)
 |-- _c12: double (nullable = true)
 |-- _c13: double (nullable = true)
 |-- _c14: double (nullable = true)
 |-- _c15: double (nullable = true)
 |-- _c16: double (nullable = true)
 |-- _c17: double (nullable = true)
 |-- _c18: double (nullable = true)
 |-- _c19: double (nullable = true)
 |-- _c20: double (nullable = true)
 |-- _c21: double (nullable = true)
 |-- _c22: double (nullable = true)



+-----+-----+-----+-----+------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|  _c0|  _c1|  _c2|  _c3|   _c4| _c5|  _c6|_c7|_c8|_c9|_c10| _c11|  _c12| _c13|   _c14|   _c15|   _c16|   _c17|  _c18|  _c19|   _c20|   _c21|   _c22|
+-----+-----+-----+-----+------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|36890|71480|10001|44404|493944|2898|44093| 10|294|  0|  31| 18.0| 30.78|23.08|  238.7| 715.48|  558.0| 954.18| 21.46|   0.0| 715.48| 736.94| 157.48|
|36890|71480|10000|44404|493944|2898|44093| 10|279|  0|  54|99.21|148.82|83.34|3535.92|4500.36|5357.34|8036.28|225.02|   0.0|4500.36|4725.38|-856.98|
|36890|71480|15556|44404|493944|2898|44093| 10|114|  0|  39|79.49| 94.59|83.24| 442.65|3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56|
+-----+-----+-----+-----+------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-----

**Constat :** Spark lit bien 23 colonnes, mais il les nomme `_c0`, `_c1`… (pas d'en-tête) et il **devine** les
types : les montants deviennent des `double` (nombres à virgule flottante, source d'erreurs d'arrondi), les
identifiants des `int`. L'inférence est une aide pour démarrer, **pas une validation du schéma**.

→ On applique donc les noms de colonnes du schéma TPCx-BB vérifié au Jour 1 (`schema_tpcxbb.py`), et on lit
d'abord tout en texte pour contrôler nous-mêmes chaque conversion.

In [3]:
ventes_txt = U.lire_bronze(spark, "store_sales")      # lots initial + refresh, colonnes de provenance ajoutées
df_ventes, echecs = U.typer(ventes_txt, "store_sales")   # conversion explicite vers les types de la spécification
print("Valeurs non convertibles :", echecs or "aucune")
df_ventes.printSchema()

Valeurs non convertibles : aucune
root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: decimal(7,2) (nullable = true)
 |-- ss_n

In [4]:
df_ventes.select("ss_sold_date_sk", "ss_item_sk", "ss_ticket_number", "ss_quantity", "ss_sales_price",
                 "ss_ext_sales_price", "ss_net_profit", "lot_source").show(10, truncate=False)
print("Nombre de lignes :", df_ventes.count())
df_ventes.groupBy("lot_source").count().show()

+---------------+----------+----------------+-----------+--------------+------------------+-------------+----------+
|ss_sold_date_sk|ss_item_sk|ss_ticket_number|ss_quantity|ss_sales_price|ss_ext_sales_price|ss_net_profit|lot_source|
+---------------+----------+----------------+-----------+--------------+------------------+-------------+----------+
|36890          |10001     |0               |31         |23.08         |715.48            |157.48       |initial   |
|36890          |10000     |0               |54         |83.34         |4500.36           |-856.98      |initial   |
|36890          |15556     |0               |39         |83.24         |3246.36           |-470.56      |initial   |
|36890          |4597      |1               |20         |133.40        |2668.00           |1180.40      |initial   |
|36890          |3258      |1               |76         |11.85         |900.60            |153.52       |initial   |
|36890          |15572     |1               |18         |30.03  

Nombre de lignes : 674211


+----------+------+
|lot_source| count|
+----------+------+
|   initial|667579|
|   refresh|  6632|
+----------+------+



**Lecture :** une ligne = **un article sur un ticket de caisse** (grain « ligne de ticket »). 674 211 lignes au
total : 667 579 dans le lot initial, 6 632 dans le lot de rafraîchissement (fin décembre 2005).

## 2. Valeurs manquantes (exemple guidé du TP §4.6.2)

In [5]:
df_ventes.cache()
for nom_colonne in U.SCHEMA["store_sales"]:
    nb_vides = df_ventes.filter(F.col(nom_colonne).isNull()).count()
    if nb_vides:
        print(f"{nom_colonne:22s}: {nb_vides:4d} valeur(s) nulle(s)")
print("(les colonnes non listées n'ont aucune valeur nulle)")

ss_cdemo_sk           :  250 valeur(s) nulle(s)


ss_hdemo_sk           :  308 valeur(s) nulle(s)


ss_addr_sk            :  462 valeur(s) nulle(s)


ss_promo_sk           :  159 valeur(s) nulle(s)


(les colonnes non listées n'ont aucune valeur nulle)


**Constat :** les colonnes indispensables au chiffre d'affaires (date, article, ticket, quantité, montants)
sont **complètes**. Les quelques nulls (0,02 à 0,07 %) concernent des colonnes facultatives : promotion,
données démographiques, adresse. Une vente sans promotion est une **absence légitime** : on ne remplace pas
ces nulls par 0, on les conserve (règle R10).

## 3. Doublons et grain

In [6]:
cols = U.SCHEMA["store_sales"]
n = df_ventes.count()
print("Doublons exacts (toutes colonnes identiques) :", n - df_ventes.select(*cols).distinct().count())
k = df_ventes.select("ss_ticket_number", "ss_item_sk").distinct().count()
print("Lignes en trop sur (ticket, article)        :", n - k)
(df_ventes.groupBy("ss_ticket_number", "ss_item_sk").count().filter("count > 1")
 .join(df_ventes, ["ss_ticket_number", "ss_item_sk"])
 .select("ss_ticket_number", "ss_item_sk", "ss_quantity", "ss_sales_price", "ss_ext_sales_price")
 .orderBy("ss_ticket_number", "ss_item_sk").show(6))

Doublons exacts (toutes colonnes identiques) : 0


Lignes en trop sur (ticket, article)        : 4182


+----------------+----------+-----------+--------------+------------------+
|ss_ticket_number|ss_item_sk|ss_quantity|ss_sales_price|ss_ext_sales_price|
+----------------+----------+-----------+--------------+------------------+
|              44|     10003|         31|         53.16|           1647.96|
|              44|     10003|         46|          1.60|             73.60|
|              46|     10000|         20|         29.71|            594.20|
|              46|     10000|         63|         36.12|           2275.56|
|              46|     17502|         55|         13.03|            716.65|
|              46|     17502|         44|         69.24|           3046.56|
+----------------+----------+-----------+--------------+------------------+
only showing top 6 rows


**Constat :** aucune ligne n'est un doublon exact, mais le couple (ticket, article) n'est **pas unique** : un même
article peut apparaître sur deux lignes d'un ticket, avec une quantité et un prix différents. Ce ne sont donc
**pas** des doublons à supprimer : ce sont deux lignes légitimes. Le fichier n'a pas de numéro de ligne, il faut
**créer une clé technique** en Silver (règle R4).

## 4. Valeurs atypiques

In [7]:
df_ventes.select("ss_quantity", "ss_sales_price", "ss_ext_sales_price", "ss_net_profit").summary(
    "count", "min", "25%", "50%", "75%", "max").show()
neg = df_ventes.filter("ss_net_profit < 0").count()
print(f"Lignes à marge négative : {neg} ({100 * neg / n:.1f} %)")
print("Prix de vente > prix catalogue :", df_ventes.filter("ss_sales_price > ss_list_price").count())
print("Écart ext_sales_price vs prix x quantité > 0,01 :",
      df_ventes.filter("abs(ss_ext_sales_price - ss_sales_price * ss_quantity) > 0.01").count())

+-------+-----------+--------------+------------------+-------------+
|summary|ss_quantity|ss_sales_price|ss_ext_sales_price|ss_net_profit|
+-------+-----------+--------------+------------------+-------------+
|  count|     674211|        674211|            674211|       674211|
|    min|          1|          0.00|              0.00|     -9866.60|
|    25%|         26|         10.24|             313.5|     -1462.37|
|    50%|         51|         27.69|           1048.22|      -377.35|
|    75%|         76|         56.77|            2674.2|         5.82|
|    max|        100|        199.30|          18745.00|      9134.00|
+-------+-----------+--------------+------------------+-------------+



Lignes à marge négative : 500889 (74.3 %)


Prix de vente > prix catalogue : 0


Écart ext_sales_price vs prix x quantité > 0,01 : 0


**Constat :** les quantités vont de 1 à 100, les montants sont cohérents (montant de ligne = prix × quantité,
0 écart). Certains prix de vente valent 0,00 : avec une remise pouvant atteindre 100 %, c'est un cas prévu par le générateur, pas une erreur. En revanche, **74 % des lignes ont une marge négative**. Est-ce une erreur ? Non : la spécification
TPCx-BB génère des remises de 0 à 100 % du prix catalogue, ce qui fait souvent passer le prix sous le coût
d'achat (constat P3/P4 du Jour 1). Une valeur surprenante n'est pas forcément fausse : ces lignes sont
**conservées** et signalées (`marge_negative`), et la marge est déclarée **non interprétable** dans les jeux Gold.

## 5. Deuxième source : les produits (`item`)

In [8]:
df_produits, echecs_i = U.typer(U.lire_bronze(spark, "item"), "item")
print("Valeurs non convertibles :", echecs_i or "aucune")
df_produits.select("i_item_sk", "i_item_id", "i_category", "i_class", "i_brand", "i_current_price",
                   "i_rec_start_date", "i_rec_end_date", "lot_source").show(5, truncate=30)
print("Lignes :", df_produits.count(), "| i_item_sk distincts :", df_produits.select("i_item_sk").distinct().count())
df_produits.groupBy("lot_source").count().show()
df_produits.groupBy("i_category").count().orderBy("i_category").show()

Valeurs non convertibles : aucune
+---------+----------------+-----------------+---------------------+------------------------------+---------------+----------------+--------------+----------+
|i_item_sk|       i_item_id|       i_category|              i_class|                       i_brand|i_current_price|i_rec_start_date|i_rec_end_date|lot_source|
+---------+----------------+-----------------+---------------------+------------------------------+---------------+----------------+--------------+----------+
|        0|AAAAAAAAAAAAAAAA|Sports & Outdoors|             Fan Shop|71TS7NSbvH1YbdiQMG6ttBHKAlj...|          72.29|      2000-01-14|          NULL|   initial|
|        1|AAAAAAAAAAAAAAAB|            Books|                  New|               qNBimHGBu763udH|          95.67|      2000-09-23|          NULL|   initial|
|        2|AAAAAAAAAAAAAAAC|            Books|Education & Reference|                           ma5|          62.74|      2000-09-01|          NULL|   initial|
|        3|A

Lignes : 18000 | i_item_sk distincts : 18000


+----------+-----+
|lot_source|count|
+----------+-----+
|   initial|17820|
|   refresh|  180|
+----------+-----+



+--------------------+-----+
|          i_category|count|
+--------------------+-----+
|               Books| 5386|
|Clothing & Access...|  617|
|         Electronics|  701|
|      Home & Kitchen|  482|
|         Movies & TV| 1300|
|               Music| 8224|
|   Sports & Outdoors|  466|
|Tools & Home Impr...|  413|
|        Toys & Games|  411|
+--------------------+-----+



**Constat :** 18 000 produits (17 820 dans le lot initial + 180 nouveaux dans le refresh), `i_item_sk` unique et
sans null : c'est la **clé** du référentiel. 9 catégories, aucune catégorie manquante (Music et Books représentent 76 % des produits). `i_rec_end_date` est
vide à 100 % : les produits ne sont pas historisés (une seule version par produit).

## 6. Avant de joindre : la relation ventes → produits est-elle vérifiée ?

Ne pas joindre sur la seule ressemblance des noms (`ss_item_sk` / `i_item_sk`). On vérifie : types identiques,
clé unique côté produit, et aucune vente sans produit correspondant.

In [9]:
print("Types :", dict(df_ventes.dtypes)["ss_item_sk"], "/", dict(df_produits.dtypes)["i_item_sk"])
orphelines = df_ventes.join(df_produits, df_ventes.ss_item_sk == df_produits.i_item_sk, "left_anti").count()
print("Ventes sans produit correspondant :", orphelines)
avant = df_ventes.count()
apres = df_ventes.join(df_produits.select("i_item_sk", "i_category"),
                       df_ventes.ss_item_sk == df_produits.i_item_sk, "left").count()
print("Lignes avant jointure :", avant, "| après :", apres)

Types : bigint / bigint


Ventes sans produit correspondant : 0


Lignes avant jointure : 674211 | après : 674211


**Conclusion :** relation **N:1 vérifiée** (types BIGINT des deux côtés, clé produit unique, 0 orpheline, nombre de
lignes conservé). La jointure ventes × produits peut être utilisée en Gold sans risque de multiplier les lignes.

## 7. Synthèse des 8 sources (profilage automatique `02_inventaire.py`)

In [10]:
import pandas as pd
inv = pd.read_csv(C.PREUVES / "exploration" / "inventaire_sources.csv", sep=";")
inv[["table", "lignes", "lignes_initial", "lignes_refresh", "colonnes", "cles_candidates",
     "cle_metier_controlee", "doublons_exacts", "colonnes_avec_nulls", "valeurs_non_convertibles"]]

,table,lignes,lignes_initial,lignes_refresh,colonnes,cles_candidates,cle_metier_controlee,doublons_exacts,colonnes_avec_nulls,valeurs_non_convertibles
0,store_sales,674211,667579,6632,23,aucune colonne seule,ss_ticket_number+ss_item_sk : 4182 lignes en trop,0,4,0
1,web_sales,674931,668052,6879,34,aucune colonne seule,ws_order_number+ws_item_sk : 4115 lignes en trop,0,11,0
2,store_returns,38328,37902,426,20,aucune colonne seule,sr_ticket_number+sr_item_sk : 160 lignes en trop,0,4,0
3,web_returns,38887,38487,400,24,aucune colonne seule,wr_order_number+wr_item_sk : 153 lignes en trop,0,8,0
4,item,18000,17820,180,22,"i_item_sk, i_item_id, i_product_name",i_item_sk : unique,0,4,0
5,store,12,12,0,29,"s_store_sk, s_store_id, s_closed_date_sk, s_st...",s_store_sk : unique,0,1,0
6,date_dim,109573,109573,0,28,"d_date_sk, d_date_id, d_date",d_date_sk : unique,0,0,0
7,reason,35,35,0,3,"r_reason_sk, r_reason_id, r_reason_desc",r_reason_sk : unique,0,0,0


## Conclusions de l'exploration

| Question du TP (§4.6.3) | Réponse observée |
|---|---|
| Colonnes numériques | quantités (entiers), montants (`DECIMAL(7,2)`), clés `*_sk` (entiers longs) |
| Colonnes de date | aucune date « en clair » dans les faits : la date est une clé (`ss_sold_date_sk`) vers `date_dim.d_date` |
| Identifiants | `*_sk` (clés techniques), `*_id` (identifiants métier texte), numéro de ticket / commande |
| Valeurs manquantes | uniquement sur des colonnes facultatives (0,02 à 0,16 %) ; 100 % sur `i_rec_end_date` / `s_rec_end_date` |
| Doublons | aucun doublon exact ; (ticket, article) non unique = deux lignes légitimes → clé technique |
| Valeurs incohérentes | aucune incohérence de calcul ; marges négatives massives mais expliquées par le générateur |

Ces constats alimentent les **règles de nettoyage** (R1 à R13) appliquées par `03_silver.py`.

In [11]:
df_ventes.unpersist()
spark.stop()